# nb26 - H1 test: the clean all-cells floor

Error-budget equations (see session log): sigma_eff^2 = stoch^2 + leak(W)^2 + pileup(W)^2 + model^2. The intrinsic (stochastic) floor at our median energy is ~0.023 (design ~10%/sqrt(E) (+) 1%), yet clean kNN-25 measures 0.0403 - so ~0.033 of error on clean is **leakage + estimator**, reducible in principle. **H1: give the model ALL cells on clean (no pileup) and sigma_eff should drop toward ~0.025-0.035.** If it does, the road to 0.03 on min-bias is physically open (then attack pileup via template decomposition, H2). If it stays ~0.040, the loss is not leakage and the budget needs revisiting.

Anchors: clean kNN-25 MeanResidual 0.0403, PairT 0.0413 (nb19, MSE). Here: all cells (cap 240), Huber, 2 seeds.

In [1]:
import os, sys, pathlib, copy, time
import numpy as np, pandas as pd, uproot, awkward as ak, matplotlib.pyplot as plt
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS, CELL_KEYS
FILES = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
OUT = REPO / 'reports' / 'predictions'; OUT.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODE = os.environ.get('NB26_MODE', 'full')
print('device', DEVICE, '| mode', MODE, '|', len(FILES), 'clean files')

device cuda | mode full | 100 clean files


In [2]:
TKEYS = CELL_KEYS + ['cell_times_front', 'cell_times_back']
AUX = ['sig_flux_prod_vertex_z', 'sig_flux_eTot']
LCAP = 240
def build_all(files, vertex_max=100.0):
    O = {k: [] for k in ['tok', 'agg', 'y', 'Etrue']}
    nfull = 0; ntot = 0
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(TKEYS + AUX, library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        for i in np.flatnonzero(vz < vertex_max):
            cc = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TKEYS}
            e = cc['energy']
            if len(e) == 0: continue
            seed = int(np.argmax(e))
            x = cc['cell_x']; yy = cc['cell_y']; ix = cc['imodx']; iy = cc['jmody']
            pts = np.stack([x, yy], 1)
            pitch = np.full(len(x), np.nan)
            for key in {(int(p), int(q)) for p, q in zip(ix, iy)}:
                sel = (ix == key[0]) & (iy == key[1]); p = pts[sel]
                if len(p) >= 2:
                    d = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf
                    pitch[sel] = np.median(np.min(d, axis=1))
            fill = np.nanmedian(pitch) if np.isfinite(pitch).any() else 120.0
            pitch[~np.isfinite(pitch)] = fill
            mod = np.array([int(np.argmin(np.abs(PITCH - p))) for p in pitch], dtype=int)
            rx = x - x[seed]; ry = yy - yy[seed]; rdr = np.hypot(rx, ry)
            o = np.argsort(rdr); ntot += 1; nfull += int(len(e) <= LCAP)
            o = o[:LCAP]
            e2, fr, bk = e[o], cc['cell_energies_front'][o], cc['cell_energies_back'][o]
            rx, ry, rdr, pit, md_ = rx[o], ry[o], rdr[o], pitch[o], mod[o]
            cont = np.stack([np.log1p(np.clip(e2, 0, None)), np.log1p(np.clip(fr, 0, None)),
                             np.log1p(np.clip(bk, 0, None)), rx / pit, ry / pit, rdr / pit,
                             np.log(pit)], 1)
            oh = np.zeros((len(e2), len(PITCH))); oh[np.arange(len(e2)), md_] = 1.0
            O['tok'].append(np.concatenate([cont, oh], 1).astype(np.float32))
            sumE = float(e2.sum()); seedE = float(e2[0])
            lat = float(np.sqrt((e2 * rdr ** 2).sum() / (sumE + EPS)))
            fb = float(fr.sum() / (bk.sum() + EPS))
            O['agg'].append([np.log1p(sumE), fb, len(e2), np.log1p(seedE), lat, int(md_[0])])
            et = float(a['sig_flux_eTot'][i])
            O['y'].append(np.log(max(et, 1e-3))); O['Etrue'].append(et)
    for k in ['agg', 'y', 'Etrue']: O[k] = np.array(O[k])
    print(f'clusters {ntot} | fully contained at LCAP={LCAP}: {100*nfull/max(ntot,1):.1f}%')
    return O

## Models: MeanResidual (clean champion) + PairT (space), both Huber

In [3]:
CFG = dict(d=96, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96, pair_hidden=32, huber_delta=0.1)
N_GLOBAL = 5
def encoder():
    layer = nn.TransformerEncoderLayer(CFG['d'], CFG['nhead'], dim_feedforward=4*CFG['d'],
                                       dropout=CFG['dropout'], batch_first=True)
    return nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
def mlp_head(nf):
    return nn.Sequential(nn.Linear(nf, CFG['d']), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(CFG['d'], 1))
class MeanResidual(nn.Module):
    def __init__(self, in_dim):
        super().__init__(); self.embed = nn.Linear(in_dim, CFG['d']); self.enc = encoder()
        self.norm = nn.LayerNorm(CFG['d']); self.head = mlp_head(CFG['d'] + N_GLOBAL)
    def forward(self, x, m, w, g, base, R=None):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))
def pair_features(R):
    rx, ry, le = R[..., 0], R[..., 1], R[..., 2]
    dx = rx.unsqueeze(2) - rx.unsqueeze(1); dy = ry.unsqueeze(2) - ry.unsqueeze(1)
    dR = torch.sqrt(dx*dx + dy*dy + 1e-6)
    esum = le.unsqueeze(2) + le.unsqueeze(1); emin = torch.minimum(le.unsqueeze(2), le.unsqueeze(1))
    return torch.stack([dx, dy, dR, esum, emin], -1)
class PairEmbed(nn.Module):
    def __init__(self):
        super().__init__()
        h = CFG['pair_hidden']
        self.net = nn.Sequential(nn.Linear(5, h), nn.GELU(), nn.Linear(h, h), nn.GELU(), nn.Linear(h, CFG['nhead']))
    def forward(self, pf): return self.net(pf).permute(0, 3, 1, 2).contiguous()
class PMHA(nn.Module):
    def __init__(self, d, nh, drop):
        super().__init__(); self.h = nh; self.dh = d // nh
        self.q = nn.Linear(d, d); self.k = nn.Linear(d, d); self.v = nn.Linear(d, d); self.o = nn.Linear(d, d)
        self.drop = nn.Dropout(drop)
    def forward(self, x, U, kv):
        B, L, d = x.shape
        q = self.q(x).view(B, L, self.h, self.dh).transpose(1, 2)
        k = self.k(x).view(B, L, self.h, self.dh).transpose(1, 2)
        v = self.v(x).view(B, L, self.h, self.dh).transpose(1, 2)
        s = (q @ k.transpose(-2, -1)) / (self.dh ** 0.5) + U
        s = s.masked_fill((~kv).view(B, 1, 1, L), -1e9)
        return self.o((self.drop(s.softmax(-1)) @ v).transpose(1, 2).reshape(B, L, d))
class Block(nn.Module):
    def __init__(self, d, nh, drop):
        super().__init__(); self.n1 = nn.LayerNorm(d); self.attn = PMHA(d, nh, drop); self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Dropout(drop), nn.Linear(4*d, d))
        self.drop = nn.Dropout(drop)
    def forward(self, x, U, kv):
        x = x + self.drop(self.attn(self.n1(x), U, kv)); return x + self.drop(self.ff(self.n2(x)))
class PairT(nn.Module):
    def __init__(self, in_dim):
        super().__init__(); d = CFG['d']
        self.embed = nn.Linear(in_dim, d); self.pair = PairEmbed()
        self.blocks = nn.ModuleList([Block(d, CFG['nhead'], CFG['dropout']) for _ in range(CFG['layers'])])
        self.norm = nn.LayerNorm(d); self.head = mlp_head(d + N_GLOBAL)
    def forward(self, x, m, w, g, base, R):
        U = self.pair(pair_features(R)); h = self.embed(x)
        for blk in self.blocks: h = blk(h, U, m)
        p = self.norm((h * w.unsqueeze(-1)).sum(1))
        return base + self.head(torch.cat([p, g], 1))

## Prep (GPU-resident tensors, batch 96 - stability mitigations)

In [4]:
O = build_all(FILES if MODE == 'full' else FILES[:8])
N = len(O['Etrue']); Et = O['Etrue']; y = O['y'].astype(np.float32); agg = O['agg']
keep = np.flatnonzero((Et >= 1.0) & (Et <= 100.0))
ktr, kva, kte = (keep[s] for s in split(len(keep)))
G = np.stack([agg[:,0], agg[:,3], np.log(agg[:,2]+1.0), agg[:,1], agg[:,4]], 1).astype(np.float32)
G = (G - G[ktr].mean(0)) / (G[ktr].std(0) + EPS)
la, lb = np.polyfit(agg[ktr,0], y[ktr], 1); base_all = (la*agg[:,0] + lb).astype(np.float32)
maxL = max(t.shape[0] for t in O['tok'])
IN_DIM = O['tok'][0].shape[1]
X = np.zeros((N, maxL, IN_DIM), np.float32); M = np.zeros((N, maxL), np.bool_); W = np.zeros((N, maxL), np.float32)
Rn = np.zeros((N, maxL, 3), np.float32)
for i, t in enumerate(O['tok']):
    L = t.shape[0]; X[i, :L] = t; M[i, :L] = True
    Rn[i, :L, 0] = t[:, 3]; Rn[i, :L, 1] = t[:, 4]; Rn[i, :L, 2] = t[:, 0]
    e = np.expm1(np.clip(t[:, 0], 0, None)); W[i, :L] = e / (e.sum() + 1e-9)
cont = X[ktr][:, :, :7].reshape(-1, 7)[M[ktr].reshape(-1)]
mean = cont.mean(0); std = cont.std(0) + EPS
X[:, :, :7] = (X[:, :, :7] - mean) / std; X[~M] = 0.0
Xc = torch.from_numpy(X).to(DEVICE); Mc = torch.from_numpy(M).to(DEVICE); Wc = torch.from_numpy(W).to(DEVICE)
Gc = torch.from_numpy(G).to(DEVICE); Bc = torch.from_numpy(base_all).unsqueeze(1).to(DEVICE)
Yc = torch.from_numpy(y).unsqueeze(1).to(DEVICE); Rc = torch.from_numpy(Rn).to(DEVICE)
print('N', N, 'maxL', maxL, 'train/val/test', len(ktr), len(kva), len(kte))

clusters 36852 | fully contained at LCAP=240: 79.9%


N 36852 maxL 240 train/val/test 23431 5021 5021


In [5]:
pe_sum = np.exp(base_all[kte])
te_e = Et[kte]
s_sum = resolution(pe_sum, te_e)['sigma_eff']
print('=== estimator-free H1 check: calibrated ALL-CELL SUM on clean ===')
print(f'sigma_eff = {s_sum:.4f}   (kNN-25 sum anchor: clean 0.0822 | intrinsic ~0.023)')
edges = np.quantile(te_e, np.linspace(0, 1, 7))
for i in range(6):
    hi = edges[i+1] + (1e-9 if i == 5 else 0)
    mm = (te_e >= edges[i]) & (te_e < hi)
    if mm.sum() >= 20:
        print(f'  E {edges[i]:6.1f}-{edges[i+1]:6.1f} GeV: {resolution(pe_sum[mm], te_e[mm])["sigma_eff"]:.4f}  (n={int(mm.sum())})')

=== estimator-free H1 check: calibrated ALL-CELL SUM on clean ===
sigma_eff = 0.0815   (kNN-25 sum anchor: clean 0.0822 | intrinsic ~0.023)
  E    1.3-  11.0 GeV: 0.0567  (n=837)
  E   11.0-  17.6 GeV: 0.0382  (n=837)
  E   17.6-  24.4 GeV: 0.0332  (n=836)
  E   24.4-  35.1 GeV: 0.0319  (n=837)
  E   35.1-  54.0 GeV: 0.0282  (n=837)
  E   54.0-  99.9 GeV: 0.0314  (n=837)


In [6]:
x = agg[:, 0]
cf = np.polyfit(x[ktr], y[ktr], 3)
pe3 = np.exp(np.polyval(cf, x[kte]))
print('cubic-calibrated all-cell sum (deployable):', resolution(pe3, te_e)['sigma_eff'])
for i in range(6):
    hi = edges[i+1] + (1e-9 if i == 5 else 0)
    mm = (te_e >= edges[i]) & (te_e < hi)
    if mm.sum() >= 20:
        print(f'  E {edges[i]:6.1f}-{edges[i+1]:6.1f} GeV: {resolution(pe3[mm], te_e[mm])["sigma_eff"]:.4f}')
tr_e = Et[ktr]
res_pool = []
for i in range(6):
    hi = edges[i+1] + (1e-9 if i == 5 else 0)
    mtr = (tr_e >= edges[i]) & (tr_e < hi); mte = (te_e >= edges[i]) & (te_e < hi)
    if mtr.sum() < 50 or mte.sum() < 20: continue
    a1, b1 = np.polyfit(x[ktr][mtr], y[ktr][mtr], 1)
    pb = np.exp(a1 * x[kte][mte] + b1)
    res_pool.append((pb - te_e[mte]) / te_e[mte])
r = np.concatenate(res_pool); rs = np.sort(r); n = len(rs); k = max(1, int(np.ceil(0.683 * n)))
w = rs[k-1:] - rs[:n-k+1]
print('oracle per-bin-calibrated pooled floor:', round(float(w.min()/2), 4))

cubic-calibrated all-cell sum (deployable): 0.0593
  E    1.3-  11.0 GeV: 0.0797
  E   11.0-  17.6 GeV: 0.0398
  E   17.6-  24.4 GeV: 0.0388
  E   24.4-  35.1 GeV: 0.0365
  E   35.1-  54.0 GeV: 0.0325
  E   54.0-  99.9 GeV: 0.0773
oracle per-bin-calibrated pooled floor: 0.1312


In [7]:
mb = pd.read_csv(OUT / 'minbias__GateHuber.csv')
seeds = sorted(mb['seed'].unique())
blocks = [mb[mb.seed == s].reset_index(drop=True) for s in seeds]
nmb = min(len(b) for b in blocks)
mb_true = blocks[0]['true_energy'].to_numpy()[:nmb]
mb_ens = np.stack([b['pred_energy'].to_numpy()[:nmb] for b in blocks]).mean(0)
print('=== gap per E-bin: minbias GateHuber(kNN-25, seed-ens) vs clean all-cell floor ===')
print(f'{"E bin [GeV]":>16s} {"minbias":>8s} {"clean floor":>11s} {"gap(pileup)":>11s}')
for i in range(6):
    hi = edges[i+1] + (1e-9 if i == 5 else 0)
    mm = (mb_true >= edges[i]) & (mb_true < hi)
    mc = (te_e >= edges[i]) & (te_e < hi)
    if mm.sum() < 20 or mc.sum() < 20: continue
    smb = resolution(mb_ens[mm], mb_true[mm])['sigma_eff']
    scl = resolution(pe3[mc], te_e[mc])['sigma_eff']
    print(f'{edges[i]:7.1f}-{edges[i+1]:6.1f} {smb:8.4f} {scl:11.4f} {np.sqrt(max(smb**2 - scl**2, 0)):11.4f}')

=== gap per E-bin: minbias GateHuber(kNN-25, seed-ens) vs clean all-cell floor ===
     E bin [GeV]  minbias clean floor gap(pileup)
    1.3-  11.0   0.0763      0.0797      0.0000
   11.0-  17.6   0.0489      0.0398      0.0284
   17.6-  24.4   0.0376      0.0388      0.0000
   24.4-  35.1   0.0379      0.0365      0.0102
   35.1-  54.0   0.0391      0.0325      0.0217
   54.0-  99.9   0.0418      0.0773      0.0000


## Train / eval (Huber)

In [8]:
EPOCHS = {'smoke': 3, 'full': 120}[MODE]
PATIENCE = {'smoke': 99, 'full': 20}[MODE]
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
def train_eval(kind, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = (MeanResidual(IN_DIM) if kind == 'MeanResidual' else PairT(IN_DIM)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    def fwd(b):
        return model(Xc[b], Mc[b], Wc[b], Gc[b], Bc[b], Rc[b])
    def lossf(p, b): return nn.functional.huber_loss(p, Yc[b], delta=CFG['huber_delta'])
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def run(idx):
        out = []
        model.eval()
        with torch.no_grad():
            for b in batches(idx, 256, False): out.append(fwd(b).cpu().numpy().ravel())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; n = 0
        with torch.no_grad():
            for b in batches(kva, 256, False): s += lossf(fwd(b), b).item(); n += 1
        return s / max(n, 1)
    best = 1e9; bstate = None; wait = 0
    for ep in range(EPOCHS):
        model.train()
        for b in batches(ktr, CFG['batch'], True):
            opt.zero_grad(); lossf(fwd(b), b).backward(); opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= PATIENCE: break
    model.load_state_dict(bstate)
    a, b2 = np.polyfit(run(kva), y[kva], 1)
    pe = np.exp(a * run(kte) + b2)
    return float(resolution(pe, Et[kte])['sigma_eff']), pe

## Run (resumable) + per-energy-bin check

In [9]:
CSVP = OUT / 'nb26_clean_floor.csv'
done = set()
if CSVP.exists():
    prev = pd.read_csv(CSVP); done = set(zip(prev['config'], prev['seed']))
    print('resume, done:', sorted(done))
PREDS = {}
for kind in ['MeanResidual']:
    for seed in SEEDS:
        if (kind, seed) in done: print('skip', kind, seed); continue
        t0 = time.time()
        sig, pe = train_eval(kind, seed)
        PREDS[(kind, seed)] = pe
        row = dict(config=kind, seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t0))
        pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
        print(f'{kind} seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
RES = pd.read_csv(CSVP); print(RES.to_string(index=False))

resume, done: [('MeanResidual', 0), ('MeanResidual', 1)]
skip MeanResidual 0
skip MeanResidual 1
      config  seed  sigma_eff  elapsed
MeanResidual     0     0.0428      683
MeanResidual     1     0.0412      890


In [10]:
print('=== H1 verdict ===')
print('anchors: clean kNN-25 MeanResidual 0.0403 | PairT 0.0413 | intrinsic ~0.023')
for kind in ['MeanResidual', 'PairT']:
    sub = RES[RES.config == kind]
    if len(sub): print(f'{kind} all-cells: {sub.sigma_eff.mean():.4f} +/- {sub.sigma_eff.std():.4f}')
best_kind = RES.groupby('config')['sigma_eff'].mean().idxmin()
ens = [p for (k, s), p in PREDS.items() if k == best_kind]
if len(ens) >= 2:
    print(f'{best_kind} seed-ensemble:', resolution(np.stack(ens).mean(0), Et[kte])['sigma_eff'])
pe = PREDS.get((best_kind, 0))
if pe is not None:
    te_e = Et[kte]
    edges = np.quantile(te_e, np.linspace(0, 1, 7))
    print('per-E-bin sigma_eff (' + best_kind + '):')
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        if mm.sum() >= 20:
            print(f'  E {edges[i]:6.1f}-{edges[i+1]:6.1f} GeV: {resolution(pe[mm], te_e[mm])["sigma_eff"]:.4f}  (n={int(mm.sum())})')

=== H1 verdict ===
anchors: clean kNN-25 MeanResidual 0.0403 | PairT 0.0413 | intrinsic ~0.023
MeanResidual all-cells: 0.0420 +/- 0.0011
